In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import torch
torch.cuda.is_available()

True

# Generate data

In [12]:
import random

NUM_ITEMS = 8  # item ids will be 1..NUM_ITEMS -- 0 stays reserved for padding
BASE_PRICE = {item_id: random.randint(500, 3000) for item_id in range(1, NUM_ITEMS + 1)}

def generate_customer(min_events=3, max_events=8, min_age=20, max_age=60):
    n_events = random.randint(min_events, max_events)
    age = random.randint(min_age, max_age)

    items, ages, prices = [], [], []
    for _ in range(n_events):
        item_id = random.randint(1, NUM_ITEMS)
        price = BASE_PRICE[item_id] * random.lognormvariate(0, 0.15)

        items.append(item_id)
        ages.append(age)
        prices.append(round(price, 2))

        age += random.randint(0, 3)  # customer ages a bit between purchases

    return items, ages, prices

def generate_dataset(n_customers=500, min_events=3, max_events=8):
    return [generate_customer(min_events, max_events) for _ in range(n_customers)]

In [13]:
samples = generate_dataset()
len(samples)

500

In [14]:
samples[0], samples[-1]

(([8, 3, 5], [57, 59, 61], [3013.34, 2974.62, 1655.12]),
 ([7, 7, 5, 5, 3, 6, 4],
  [31, 34, 35, 37, 40, 42, 45],
  [2320.36, 2299.86, 2165.46, 2453.09, 2577.94, 1597.62, 2682.86]))

# Start!

In [15]:
import numpy as np
import pandas as pd

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Create Dataset/DataLoader with custom collate_fn

In [17]:
class SampleDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self, ):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

In [18]:
dataset = SampleDataset(samples=samples)
len(dataset), dataset[0], dataset[-1]

(500,
 ([8, 3, 5], [57, 59, 61], [3013.34, 2974.62, 1655.12]),
 ([7, 7, 5, 5, 3, 6, 4],
  [31, 34, 35, 37, 40, 42, 45],
  [2320.36, 2299.86, 2165.46, 2453.09, 2577.94, 1597.62, 2682.86]))

In [19]:
def next_item_collate_fn(batch):
    """What we do is to create: x, y which x is seq 0 to max-1 and y is -1"""
    items, ages, prices, next_items, next_ages, next_prices = [], [], [], [], [], []
    for item, age, price in batch:
        items.append(item[:-1])
        ages.append(age[:-1])
        prices.append(price[:-1])
        next_items.append(item[-1])
        next_ages.append(age[-1])
        next_prices.append(price[-1])
    return (
        items, ages, prices, next_items, next_ages, next_prices
    )

In [20]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=None)
next_item_dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=next_item_collate_fn)

In [21]:
items, ages, prices, next_items, next_ages, next_prices = next_item_collate_fn(batch=[dataset[i] for i in range(4)])

In [22]:
items, next_items

([[8, 3], [1, 6, 4], [1, 2, 6, 3, 7, 4], [2, 2, 2, 4, 4]], [5, 5, 5, 7])

In [23]:
dataset[2]

([1, 2, 6, 3, 7, 4, 5],
 [42, 42, 44, 47, 47, 48, 48],
 [1048.46, 912.21, 1815.73, 2667.77, 2081.4, 2663.85, 2014.48])

In [24]:
batch = next(iter(next_item_dataloader))
batch

([[2, 2],
  [8, 6, 1, 4, 5, 5],
  [8, 3, 5],
  [7, 6, 7, 6, 1],
  [7, 2],
  [4, 8],
  [7, 3, 5],
  [6, 1, 3, 7],
  [1, 4, 7],
  [4, 7, 8, 5, 7, 1],
  [8, 5],
  [2, 2, 3, 3, 5],
  [4, 5, 5, 5, 5, 4, 8],
  [2, 6, 2],
  [8, 6, 2, 3, 5, 7, 1],
  [7, 8, 8],
  [6, 6, 4, 2, 5, 6],
  [2, 7, 6],
  [3, 1, 6, 1],
  [8, 6, 8, 2],
  [7, 8, 5, 4],
  [4, 4, 1, 5, 4, 2],
  [4, 8, 1, 6, 8],
  [4, 7, 6, 1, 8, 7],
  [3, 6],
  [8, 2, 5],
  [8, 2, 1],
  [2, 5, 4],
  [3, 5, 7, 8, 6, 3],
  [3, 3, 6, 5, 3, 6, 8],
  [7, 3, 7],
  [8, 6, 1]],
 [[33, 34],
  [46, 46, 47, 49, 52, 52],
  [56, 57, 60],
  [51, 54, 56, 57, 57],
  [23, 24],
  [33, 36],
  [49, 52, 52],
  [29, 32, 33, 36],
  [60, 61, 62],
  [33, 36, 36, 37, 37, 37],
  [22, 22],
  [33, 36, 39, 42, 45],
  [27, 28, 30, 31, 33, 33, 35],
  [52, 52, 53],
  [26, 27, 28, 31, 34, 37, 38],
  [26, 28, 29],
  [23, 23, 25, 28, 31, 33],
  [22, 25, 25],
  [24, 27, 29, 29],
  [51, 54, 55, 57],
  [43, 44, 46, 47],
  [59, 62, 63, 64, 65, 66],
  [41, 44, 46, 48, 51],
  [21,

In [25]:
items, ages, prices, next_item, next_age, next_price = batch

In [26]:
items[0], next_item[0]

([2, 2], 1)

In [27]:
class CategoricalEmbedding(nn.Module):
    def __init__(self, vocab_size:int, d_model:int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, X):
        return self.embedding(X)

class ContinuousEmbedding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.embedding = nn.Linear(1, d_model)

    def forward(self, X):
        return self.embedding(X)

In [28]:
d_model = 10
cat_emb = CategoricalEmbedding(10, d_model)
cont_emb = ContinuousEmbedding(d_model)

In [29]:
cat_test = torch.tensor([
    [1,2,3],
    [4,5,6],
    [7,8,9]
], dtype=torch.long)
cont_test = torch.tensor([
    [1,2,3],
    [4,5,6],
    [7,8,9]
], dtype=torch.float)

In [30]:
cat_emb(cat_test).shape

torch.Size([3, 3, 10])

In [31]:
cont_emb(cont_test.unsqueeze(-1)).shape

torch.Size([3, 3, 10])

# what we need next is to master collate function

In [ ]:
class TestDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

def test_collate_fn(batch):
    items, ages, prices, lengths = [], [], [], []
    for item, age, price in batch:
        items.append(torch.tensor(item, dtype=torch.long))
        ages.append(torch.tensor(age, dtype=torch.float))
        prices.append(torch.tensor(price, dtype=torch.float))
        lengths.append(len(item))
    items = pad_sequence(items, batch_first=True, padding_value=0)
    ages = pad_sequence(ages, batch_first=True, padding_value=0.0)
    prices = pad_sequence(prices, batch_first=True, padding_value=0.0)
    max_len = items.shape[1]
    mask = torch.arange(max_len).unsqueeze(0) < torch.tensor(lengths).unsqueeze(1)
    return items, ages, prices, mask

dataset = TestDataset(samples=samples)
loader = DataLoader(dataset, batch_size=5, shuffle=True, collate_fn=test_collate_fn)
batch = next(iter(loader))

In [60]:
batch[0]

tensor([[6, 1, 4, 7, 7, 8, 4, 0],
        [7, 8, 2, 3, 4, 7, 7, 0],
        [5, 8, 7, 0, 0, 0, 0, 0],
        [3, 6, 6, 7, 3, 5, 8, 1],
        [6, 6, 6, 3, 0, 0, 0, 0]])

In [61]:
batch[1]

tensor([[43., 43., 46., 47., 50., 52., 55.,  0.],
        [44., 47., 49., 51., 52., 52., 52.,  0.],
        [23., 25., 25.,  0.,  0.,  0.,  0.,  0.],
        [54., 55., 55., 56., 58., 59., 62., 64.],
        [22., 22., 25., 27.,  0.,  0.,  0.,  0.]])

In [62]:
batch[2]

tensor([[1929.0100, 1237.3199, 2206.5200, 1749.1400, 2825.9299, 3559.5701,
         2711.5000,    0.0000],
        [2586.4399, 2521.4500,  698.7900, 2541.9700, 2614.0200, 2103.0400,
         2634.5200,    0.0000],
        [1513.4900, 2783.7000, 2485.3799,    0.0000,    0.0000,    0.0000,
            0.0000,    0.0000],
        [2846.8899, 2022.6400, 2532.8999, 2126.5500, 2124.5701, 1558.4800,
         3097.0100, 1140.4200],
        [1924.1500, 2614.6599, 1691.7700, 3510.5300,    0.0000,    0.0000,
            0.0000,    0.0000]])

In [63]:
batch[3]

tensor([[ True,  True,  True,  True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True,  True,  True, False],
        [ True,  True,  True, False, False, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True],
        [ True,  True,  True,  True, False, False, False, False]])

In [90]:
def test_next_seq_collate_fn(batch):
    items, ages, prices, lengths = [], [], [], []
    next_items, next_ages, next_prices = [], [], []
    for item, age, price in batch:
        items.append(torch.tensor(item[:-1], dtype=torch.long))
        ages.append(torch.tensor(age[:-1], dtype=torch.float))
        prices.append(torch.tensor(price[:-1], dtype=torch.float))
        next_items.append(item[-1])
        next_ages.append(age[-1])
        next_prices.append(price[-1])
        lengths.append(len(item)-1)

    items = pad_sequence(items, batch_first=True, padding_value=0)
    ages = pad_sequence(ages, batch_first=True, padding_value=0.0)
    prices = pad_sequence(prices, batch_first=True, padding_value=0.0)
    next_items = torch.tensor(next_items, dtype=torch.long)
    next_ages = torch.tensor(next_ages, dtype=torch.float)
    next_prices = torch.tensor(next_prices, dtype=torch.float) 
    max_len = items.shape[1]
    mask = torch.arange(max_len).unsqueeze(0) < torch.tensor(lengths).unsqueeze(1)
    return items, ages, prices, next_items, next_ages, next_prices, mask

next_seq_loader = DataLoader(dataset, batch_size=5, shuffle=True, collate_fn=test_next_seq_collate_fn)
batch = next(iter(next_seq_loader))

In [91]:
len(batch)

7

In [92]:
items, ages, prices, next_items, next_ages, next_prices, mask = batch

In [93]:
mask

tensor([[ True,  True,  True,  True,  True,  True],
        [ True,  True,  True, False, False, False],
        [ True,  True,  True,  True,  True,  True],
        [ True,  True,  True, False, False, False],
        [ True,  True,  True,  True,  True, False]])

In [94]:
items

tensor([[7, 4, 4, 3, 7, 6],
        [2, 5, 2, 0, 0, 0],
        [4, 1, 3, 6, 3, 1],
        [5, 8, 4, 0, 0, 0],
        [4, 1, 4, 5, 2, 0]])

In [95]:
next_items

tensor([1, 5, 6, 6, 5])

In [101]:
ages.shape

torch.Size([5, 6])

In [100]:
ages.unsqueeze(-1).shape

torch.Size([5, 6, 1])

In [97]:
next_ages

tensor([51., 61., 46., 29., 66.])

In [102]:
wQ = nn.Linear(5,5)
wK = nn.Linear(5,5)
wV = nn.Linear(5,5)

In [103]:
emb = nn.Embedding(10, 5)
lin = nn.Linear(1, 5)

In [145]:
items = torch.randint(0,5, size=(10,5,))
items

tensor([[0, 2, 1, 3, 4],
        [3, 3, 2, 1, 1],
        [3, 0, 3, 0, 1],
        [2, 1, 1, 2, 1],
        [1, 3, 4, 2, 0],
        [4, 3, 0, 0, 0],
        [1, 0, 2, 2, 0],
        [3, 3, 1, 3, 2],
        [2, 4, 0, 2, 1],
        [1, 3, 2, 4, 4]])

In [146]:
ages = torch.randint(10, 13, size=(10, 5)).cumsum(dim=1).float()
ages

tensor([[11., 22., 34., 45., 56.],
        [10., 22., 33., 45., 55.],
        [10., 21., 33., 44., 56.],
        [12., 24., 36., 48., 58.],
        [12., 23., 33., 43., 54.],
        [12., 22., 32., 43., 54.],
        [12., 23., 33., 45., 57.],
        [12., 24., 34., 46., 56.],
        [11., 21., 33., 45., 56.],
        [12., 22., 34., 44., 55.]])

In [147]:
emb_items = emb(items)
lin_items = lin(ages.unsqueeze(-1))

In [148]:
emb_items.shape, lin_items.shape

(torch.Size([10, 5, 5]), torch.Size([10, 5, 5]))

In [149]:
concat_emb = torch.cat([emb_items, lin_items], dim=-1)
concat_emb.shape

torch.Size([10, 5, 10])

In [150]:
down_proj = nn.Linear(10, 5)

In [151]:
emb_items.shape, lin_items.shape, concat_emb.shape, down_proj(concat_emb).shape

(torch.Size([10, 5, 5]),
 torch.Size([10, 5, 5]),
 torch.Size([10, 5, 10]),
 torch.Size([10, 5, 5]))

In [152]:
x = down_proj(concat_emb)
q = wQ(x)
k = wQ(x)
v = wQ(x)

In [153]:
x.shape, q.shape, k.shape, v.shape

(torch.Size([10, 5, 5]),
 torch.Size([10, 5, 5]),
 torch.Size([10, 5, 5]),
 torch.Size([10, 5, 5]))

In [154]:
attn = q @ k.transpose(-2, -1)
attn = attn / (x.size(-1) ** 0.5)
attn.shape

torch.Size([10, 5, 5])

In [155]:
(F.softmax(attn, dim=-1) @ v).shape

torch.Size([10, 5, 5])

In [158]:
ffn = nn.Linear(5, 10)

ffn(F.softmax(attn, dim=-1) @ v).shape

torch.Size([10, 5, 10])

In [159]:
ffn = nn.Linear(5, 1)

ffn(F.softmax(attn, dim=-1) @ v).shape

torch.Size([10, 5, 1])

In [160]:
cls = nn.Parameter(torch.randn(1,1,5))
cls.shape

torch.Size([1, 1, 5])

In [161]:
cls.expand(10, -1, -1).shape

torch.Size([10, 1, 5])

# RoPE

In [ ]:
class RoPE(nn.Module):
    def __init__(self, dim: int, max_len: int):
        super().__init__()
        # speed
        theta = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))   # (dim//2,)
        positions = torch.arange(max_len).float()
        angles = torch.outer(positions, theta)              # (max_len, dim//2)
        emb = torch.cat([angles, angles], dim=-1)            # (max_len, dim) -- each freq duplicated across both halves
        self.register_buffer("cos", emb.cos())                # (max_len, dim) -- precomputed once, not per forward call
        self.register_buffer("sin", emb.sin())                # (max_len, dim)

    @staticmethod
    def rotate_half(x: torch.Tensor) -> torch.Tensor:
        x1, x2 = x.chunk(2, dim=-1)          # first half, second half
        return torch.cat((-x2, x1), dim=-1)  # no unflatten/stack -- just slice, negate, concat

    def forward(self, x: torch.Tensor, position_ids: torch.Tensor | None = None) -> torch.Tensor:
        # x: (batch, seq_len, n_heads, head_dim)
        seq_len = x.shape[1]
        if position_ids is None:
            # default: identical to the old self.freqs[:seq_len] behavior
            position_ids = torch.arange(seq_len, device=x.device)

        cos = self.cos[position_ids].unsqueeze(0).unsqueeze(2)   # (1, seq_len, 1, dim) -- broadcasts over batch & heads
        sin = self.sin[position_ids].unsqueeze(0).unsqueeze(2)

        return x * cos + self.rotate_half(x) * sin


In [166]:
dim = 6
max_len = 10
theta = 1.0 / (10_000**(torch.arange(0, dim, 2).float()/dim))
positions = torch.arange(max_len).float()

In [167]:
theta

tensor([1.0000, 0.0464, 0.0022])

In [168]:
positions

tensor([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.])

In [184]:
positions.shape, theta.shape, (positions.unsqueeze(-1) * theta).shape

(torch.Size([10]), torch.Size([3]), torch.Size([10, 3]))

In [169]:
angles = torch.outer(positions, theta)
angles

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 4.6416e-02, 2.1544e-03],
        [2.0000e+00, 9.2832e-02, 4.3089e-03],
        [3.0000e+00, 1.3925e-01, 6.4633e-03],
        [4.0000e+00, 1.8566e-01, 8.6177e-03],
        [5.0000e+00, 2.3208e-01, 1.0772e-02],
        [6.0000e+00, 2.7850e-01, 1.2927e-02],
        [7.0000e+00, 3.2491e-01, 1.5081e-02],
        [8.0000e+00, 3.7133e-01, 1.7235e-02],
        [9.0000e+00, 4.1774e-01, 1.9390e-02]])

In [170]:
theta.shape, positions.shape, angles.shape

(torch.Size([3]), torch.Size([10]), torch.Size([10, 3]))

In [171]:
emb = torch.concat([angles, angles], dim=-1)
emb.shape

torch.Size([10, 6])

In [173]:
emb.cos()

tensor([[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.9989,  1.0000,  0.5403,  0.9989,  1.0000],
        [-0.4161,  0.9957,  1.0000, -0.4161,  0.9957,  1.0000],
        [-0.9900,  0.9903,  1.0000, -0.9900,  0.9903,  1.0000],
        [-0.6536,  0.9828,  1.0000, -0.6536,  0.9828,  1.0000],
        [ 0.2837,  0.9732,  0.9999,  0.2837,  0.9732,  0.9999],
        [ 0.9602,  0.9615,  0.9999,  0.9602,  0.9615,  0.9999],
        [ 0.7539,  0.9477,  0.9999,  0.7539,  0.9477,  0.9999],
        [-0.1455,  0.9318,  0.9999, -0.1455,  0.9318,  0.9999],
        [-0.9111,  0.9140,  0.9998, -0.9111,  0.9140,  0.9998]])

In [174]:
emb.sin()

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.8415,  0.0464,  0.0022,  0.8415,  0.0464,  0.0022],
        [ 0.9093,  0.0927,  0.0043,  0.9093,  0.0927,  0.0043],
        [ 0.1411,  0.1388,  0.0065,  0.1411,  0.1388,  0.0065],
        [-0.7568,  0.1846,  0.0086, -0.7568,  0.1846,  0.0086],
        [-0.9589,  0.2300,  0.0108, -0.9589,  0.2300,  0.0108],
        [-0.2794,  0.2749,  0.0129, -0.2794,  0.2749,  0.0129],
        [ 0.6570,  0.3192,  0.0151,  0.6570,  0.3192,  0.0151],
        [ 0.9894,  0.3629,  0.0172,  0.9894,  0.3629,  0.0172],
        [ 0.4121,  0.4057,  0.0194,  0.4121,  0.4057,  0.0194]])

In [175]:
a = torch.tensor([1,2,3])
b = torch.tensor([10, 100])

In [176]:
a.shape, b.shape

(torch.Size([3]), torch.Size([2]))

In [177]:
torch.outer(a, b)

tensor([[ 10, 100],
        [ 20, 200],
        [ 30, 300]])

In [178]:
torch.outer(a, b).shape

torch.Size([3, 2])

In [180]:
a.unsqueeze(-1) * b

tensor([[ 10, 100],
        [ 20, 200],
        [ 30, 300]])

In [187]:
torch.arange(0, 10).chunk(2, dim=-1)

(tensor([0, 1, 2, 3, 4]), tensor([5, 6, 7, 8, 9]))

In [1]:
import torch
torch.cuda.is_available()

True

In [9]:
import torch.nn.functional as F
import torch.nn as nn

test_ln = nn.Linear(256, 512)
W = test_ln.weight.data
test_ln.weight.shape

torch.Size([512, 256])

In [10]:
U, S, Vh = torch.linalg.svd(W, full_matrices=False)

In [11]:
U.shape, S.shape, Vh.shape

(torch.Size([512, 256]), torch.Size([256]), torch.Size([256, 256]))

In [13]:
r = 16
U_r, S_r, Vh_r = U[:, :r], S[:r], Vh[:r, :]
U_r.shape, S_r.shape, Vh_r.shape

(torch.Size([512, 16]), torch.Size([16]), torch.Size([16, 256]))

In [14]:
W_approx = U_r @ torch.diag(S_r) @ Vh_r
W_approx.shape

torch.Size([512, 256])

In [15]:
error = (W-W_approx).norm() / W.norm()
error

tensor(0.9172)

In [16]:
A = Vh_r
B = U_r @ torch.diag(S_r)
A.shape, B.shape, (B @ A).shape

(torch.Size([16, 256]), torch.Size([512, 16]), torch.Size([512, 256]))

In [17]:
class LowRankLinear(nn.Module):
    def __init__(self, in_features, out_features, rank, bias=True):
        super().__init__()
        self.A = nn.Linear(in_features, rank, bias=False)     # weight shape (rank, in_features) == Vh_r
        self.B = nn.Linear(rank, out_features, bias=bias)     # weight shape (out_features, rank) == U_r @ diag(S_r)

    def forward(self, x):
        return self.B(self.A(x))

low_rank = LowRankLinear(256, 512, rank=16)
low_rank.A.weight.data = Vh_r
low_rank.B.weight.data = U_r @ torch.diag(S_r)


In [20]:
F.linear(torch.randn(5, 3), torch.randn(1,3))

tensor([[ 0.6150],
        [-0.7282],
        [ 0.6661],
        [-3.0890],
        [ 0.4160]])

In [23]:
torch.randn(5, 3).shape, torch.randn(1, 3).shape

(torch.Size([5, 3]), torch.Size([1, 3]))

In [24]:
torch.randn(5, 3)

tensor([[-0.4632, -1.3948, -0.3769],
        [-1.5624, -1.3736, -1.0577],
        [-0.1549, -0.0126,  1.5259],
        [ 1.2501, -0.2843, -1.2699],
        [ 0.8003,  0.5218, -1.8267]])

In [26]:
torch.randn(5, 3) @ torch.randn(1, 3).transpose(-1, 0)

tensor([[ 0.6535],
        [-0.0853],
        [-0.5074],
        [ 0.2241],
        [ 0.2069]])

In [27]:
nn.Linear(4, 3).weight.shape

torch.Size([3, 4])

In [28]:
F.linear(torch.randn(5, 4), nn.Linear(4, 1).weight.data)

tensor([[ 0.5873],
        [-0.3273],
        [-0.3720],
        [ 0.2175],
        [-1.5195]])

In [29]:
layer = nn.Linear(4, 3)          # in_features=4, out_features=3
layer.weight.shape               # torch.Size([3, 4]) -- (out, in), not (in, out)

x = torch.rand(2, 4)
y1 = F.linear(x, layer.weight, layer.bias)     # (2, 3)
y2 = x @ layer.weight.T + layer.bias           # (2, 3) -- exactly the same computation
torch.allclose(y1, y2)                          # True


True

In [38]:
nn.Embedding(10, 20).weight.data[0].shape

torch.Size([20])